In [ ]:
!pip install opencv-python ultralytics pandas pillow pyproj roboflow
!pip install numpy==1.24.4
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 147.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 160.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  


### Dependencies
Make sure to install the required Python packages in your environment before running this notebook:
```
pip install torch torchvision ultralytics pandas Pillow pyproj opencv-python
```
Adjust file paths according to your local setup; the code below assumes that any data files are accessible in the local working directory.

In [ ]:
# TensorFlow not required for YOLOv8 training
# Uncomment below if needed for other tasks
# !pip install tensorflow

In [ ]:
import torch
if torch.cuda.is_available():
    device = 0               # GPU #0 (your RTX 4060 Laptop GPU)
    print("Using GPU:", torch.cuda.get_device_name(device))
else:
    device = 'cpu'
    print("CUDA not available, using CPU")


Using GPU: NVIDIA A100-SXM4-80GB


In [ ]:
import os
from roboflow import Roboflow

# ✅ Set your Roboflow API key as an environment variable for security
# In Colab: use Secrets (left sidebar > key icon) and set ROBOFLOW_API_KEY
# Or set it manually below (not recommended for shared notebooks):
# os.environ['ROBOFLOW_API_KEY'] = 'your_key_here'


try:
    from google.colab import userdata
    api_key = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    api_key = os.environ.get('ROBOFLOW_API_KEY', '')

if not api_key:
    raise ValueError('ROBOFLOW_API_KEY not set. Add it to Colab Secrets or environment variables.')

rf = Roboflow(api_key=api_key)
project = rf.workspace('chinoye').project('crop-detection-tigp8')
version = project.version(1)
dataset = version.download('yolov8')

# ✅ Using YOLOv8 Medium — better balance of speed vs accuracy for limited datasets
YOLO_pretrained = 'yolov8m.pt'

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Crop-Detection-1 in yolov8:: 100%|██████████| 2764/2764 [00:00<00:00, 6590.61it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import csv
import os
from pathlib import Path
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS

def get_decimal_from_dms(dms, ref):
    def to_float(value):
        return float(value[0]) / float(value[1]) if isinstance(value, tuple) else float(value)
    degrees = to_float(dms[0])
    minutes = to_float(dms[1]) / 60.0
    seconds = to_float(dms[2]) / 3600.0
    decimal = degrees + minutes + seconds
    return -decimal if ref in ('S', 'W') else round(decimal, 7)

def extract_gps_info(exif_data):
    gps_info = {}
    for key, value in exif_data.items():
        if TAGS.get(key) == "GPSInfo":
            for gk, gv in value.items():
                gps_info[ GPSTAGS.get(gk, gk) ] = gv
    return gps_info

def coord_gopro(src, tgt):
    """
    Recursively extract GPS metadata from all JPEGs under `src`
    and write to the CSV file `tgt`. Parent folder of `tgt` must exist.
    """
    out_dir = os.path.dirname(tgt)
    if out_dir and not os.path.isdir(out_dir):
        raise FileNotFoundError(f"Directory '{out_dir}' does not exist.")

    with open(tgt, 'w', newline='', encoding='utf-8') as output_file:
        writer = csv.writer(output_file)
        writer.writerow(['IMG', 'Latitude', 'Longitude'])

        # recursion: find every .jpg/.jpeg under src
        for img_path in Path(src).rglob('*.jp*g'):
            try:
                with Image.open(img_path) as img:
                    exif = img._getexif() or {}
                    gps = extract_gps_info(exif)
                    if 'GPSLatitude' in gps and 'GPSLongitude' in gps:
                        lat = get_decimal_from_dms(gps['GPSLatitude'], gps['GPSLatitudeRef'])
                        lon = get_decimal_from_dms(gps['GPSLongitude'], gps['GPSLongitudeRef'])
                        writer.writerow([img_path.stem, lat, lon])
            except Exception as e:
                print(f"⚠️ Skipping {img_path.name}: {e}")
    print(f"✔️ Finished writing GPS data to: {tgt}")

# Example usage:
src = dataset.location
tgt = os.path.join(dataset.location, "Maizecoord2026.csv")
coord_gopro(src, tgt)


✔️ Finished writing GPS data to: /content/Crop-Detection-1/Maizecoord2026.csv


In [ ]:
# ###image resize###
# import os
# from PIL import Image

# # CONFIGURATION
# UNANNOTATED_DIR = "E:/IITA/Oyo23"  # root folder to search
# RESIZED_DIR      = "C:/Users/johnu/Documents/yolo25/oyo_resized_images"  # flat output folder

# # PARAMETERS
# image_size = (640, 640)  # target size

# def resize_images_flat(input_dir, output_dir, size=(640, 640)):
#     os.makedirs(output_dir, exist_ok=True)
#     count = 0

#     for root, _, files in os.walk(input_dir):
#         for file_name in files:
#             if file_name.lower().endswith((".jpg", ".jpeg", ".png")):
#                 src_path = os.path.join(root, file_name)
#                 dst_path = os.path.join(output_dir, file_name)

#                 # If you might have duplicate filenames, you could
#                 # prefix with a counter or subfolder name:
#                 # base, ext = os.path.splitext(file_name)
#                 # file_name = f"{count:05d}_{base}{ext}"
#                 # dst_path = os.path.join(output_dir, file_name)

#                 try:
#                     with Image.open(src_path) as img:
#                         img_resized = img.resize(size, Image.LANCZOS)
#                         img_resized.save(dst_path)
#                         print(f"[{count}] {src_path} → {dst_path}")
#                 except Exception as e:
#                     print(f"Error on {src_path}: {e}")
#                 count += 1

#     print(f"\nDone: {count} images resized into {output_dir}")

# if __name__ == "__main__":
#     resize_images_flat(UNANNOTATED_DIR, RESIZED_DIR, size=image_size)


In [ ]:
import os

# Dataset paths
ROOT = dataset.location
DATA_DIR = dataset.location
VALID_DIR = f'{DATA_DIR}/valid'
TEST_DIR = f'{DATA_DIR}/test'

# Output paths
project_dir = f'{ROOT}/runs'
name = 'detect'

# Trained weights path after training
MODEL_DIR = f'{project_dir}/{name}/weights/best.pt'  # ✅ Use best.pt not last.pt

# ✅ TRAINING PARAMETERS
epochs = 200
device = 0           # GPU (set to 'cpu' if no GPU available)
save_period = 5      # Save checkpoint every 5 epochs
optimizer = 'SGD'
lr0 = 0.001          # Initial learning rate
cos_lr = True        # Cosine LR scheduler for smoother convergence
patience = 30        # ✅ Early stopping: stop if no improvement for 30 epochs

print(f'Dataset: {DATA_DIR}')
print(f'Model weights will be saved to: {MODEL_DIR}')

Dataset: /content/Crop-Detection-1
Model weights will be saved to: /content/Crop-Detection-1/runs/detect/weights/best.pt


In [ ]:
import os
import cv2

from ultralytics import YOLO
import pandas as pd
import math
from PIL import Image
import csv
import pandas as pd
import math
from pyproj import Transformer
import pandas as pd

def prediction(model_dir, img_dir, pred_dir, switch_saveImg):

    model = YOLO(model_dir)

    # New file to store info
    with open(os.path.join(pred_dir, "PRED.csv"), 'w') as f:
        f.write('img,box,cls,conf,top_left_x_min,top_left_y_min,bottom_right_x_max,bottom_right_y_max\n')

    # Run batched inference on a list of images
    results = model(os.path.join(img_dir, "*jpg"))

    # Process results list
    for idx, result in enumerate(results):
        boxes = result.boxes

        if len(boxes) >= 1:
            print(f"Image {idx} - Detected boxes: {len(boxes)}")
            if switch_saveImg:
                res_plotted = result.plot()
                filename = os.path.basename(result.path)
                cv2.imwrite(os.path.join(pred_dir, filename), res_plotted)

            for i in range(len(boxes)):
                if (boxes.data[i][1] > 0.25):
                    # derive just the image ID (basename without extension)
                    fname = os.path.basename(result.path)
                    img_id = os.path.splitext(fname)[0]

                    with open(os.path.join(pred_dir, "PRED.csv"), 'a') as f:
                        f.write(
                            f'{img_id},{i+1}, '
                            f'{boxes.cls[i]},{boxes.conf[i]},'
                            f'{boxes.data[i][0]},{boxes.data[i][1]},'
                            f'{boxes.data[i][2]},{boxes.data[i][3]}\n'
                        )

def distance(pred_dir, intercroppedCLS, intercropped_thrshld):

    PRED = pd.read_csv(os.path.join(pred_dir, "PRED.csv"), header=0)

    PRED_Distance = os.path.join(pred_dir, 'PRED_Distance.csv')
    with open(PRED_Distance, 'w') as f:
        f.write("IMG,CLS,Distance,DistanceTotal,DistanceAverage\n")

    rowName_prev = ""
    for idx, row in PRED.iterrows():

        rowName = row.loc['img']
        cls = row.loc['cls']

        h_original_cashew = row.loc["bottom_right_y_max"] - row.loc["top_left_y_min"] # unit: px
        # ✅ FIX: Renamed from cashew to maize — update h_enlarged_maize and l_focal_GoPro
        # to match your actual GoPro camera calibration values before running
        l_focal_new = h_enlarged_maize / h_original_maize * l_focal_GoPro
        theta = math.atan(h_sensor / (2 * l_focal_new))  # unit: radian
        d = h_GoPro / math.tan(theta)  # distance; unit: m

        if rowName != rowName_prev:
            rowName_prev = rowName
            n = 1
            distance_total = 0
            distance_total += d
            distance_ave = distance_total
        elif rowName == rowName_prev:
            n += 1
            distance_total += d
            distance_ave = distance_total / n

        with open(PRED_Distance, 'a') as f:
            f.write(f"{rowName}, {cls}, {d}, {distance_total}, {distance_ave}\n")

    # one distance and one label for one figure
    df = pd.read_csv(PRED_Distance)
    max_distance_rows = df.sort_values('DistanceTotal', ascending=False).drop_duplicates('IMG')
    proportions = df.groupby('IMG')['CLS'].value_counts(normalize=True).unstack()# return frequency/proportion
    # print("proportions: ", proportions)
    multi_class_imgs = proportions[(proportions > intercropped_thrshld).sum(axis=1) > 1].index
    # print("multi_class_imgs: ", multi_class_imgs)
    max_distance_rows['CLS'] = max_distance_rows.apply(lambda row: intercroppedCLS if row['IMG'] in multi_class_imgs else row['CLS'], axis=1) # intercropped code: 4
    max_distance_rows.to_csv(os.path.join(pred_dir, 'PRED_DIST_CLS_unique.csv'), index=False)

def bearing(path_img_label, path_img_coord, output_folder):

    # Get coord for imgs with labels
    output_file = os.path.join(output_folder, 'bearing.csv')

    ## Read the values from the 'IMG' column in file A
    values_to_match = {}
    with open(path_img_label, mode='r', encoding='utf-8-sig') as f_a:
        reader_a = csv.DictReader(f_a)
        for row in reader_a:
            values_to_match[row["IMG"]] = row

    ## Select coord according to 'IMG' column in file A
    coord_all = {}
    columns_coord = ['Latitude', 'Longitude']
    columns_coord_assist = ['Latitude_assist', 'Longitude_assist']
    with open(path_img_coord, mode='r', encoding='utf-8-sig') as f_b:
        reader = csv.DictReader(f_b)
        for row in reader:
            # Change 'filename' to 'IMG' to match the column name in the CSV generated by coord_gopro
            coord_all[row['IMG']] = row

        for filename in values_to_match:
            if filename in coord_all:
                for col in columns_coord:
                    values_to_match[filename][col] = coord_all[filename][col]
                filename_prev = f"{filename.split('G')[0]}G{int(filename.split('G')[1]) - 2:07d}"
                filename_prev_2 = f"{filename.split('G')[0]}G{int(filename.split('G')[1]) -3 :07d}"
                if (filename_prev in coord_all):
                    if (coord_all[filename][columns_coord[0]] != coord_all[filename_prev][columns_coord[0]]) & (coord_all[filename][columns_coord[1]] != coord_all[filename_prev][columns_coord[1]]):
                        for col, col_assist in zip(columns_coord, columns_coord_assist):
                            values_to_match[filename][col_assist] = coord_all[filename_prev][col]
                elif (filename_prev_2 in coord_all):
                    if (coord_all[filename][columns_coord[0]] != coord_all[filename_prev_2][columns_coord[0]]) & (coord_all[filename][columns_coord[1]] != coord_all[filename_prev_2][columns_coord[1]]):
                        for col, col_assist in zip(columns_coord, columns_coord_assist):
                            values_to_match[filename][col_assist] = coord_all[filename_prev_2][col]
                else:
                    for col_assist in columns_coord_assist:
                        values_to_match[filename][col_assist] = None

    ## Write the updated rows to a new file
    with open(output_file, mode='w', encoding='utf-8') as f_out:
        fieldnames = reader_a.fieldnames + columns_coord + columns_coord_assist
        writer = csv.DictWriter(f_out, fieldnames=fieldnames)
        writer.writeheader()

        for row in values_to_match.values():
            writer.writerow(row)

def coord_new_points(x, y, x1, y1, distance):

    # Calculate direction vector (dx, dy)
    dx = x1 - x
    dy = y1 - y

    # Normalize the direction vector
    length = math.sqrt(dx ** 2 + dy ** 2)
    dx /= length
    dy /= length

    # Calculate perpendicular vector (to the right)
    perp_dx = dy
    perp_dy = -dx

    # Calculate new point
    new_x = x1 + perp_dx * distance
    new_y = y1 + perp_dy * distance

    return new_x, new_y

def coord_infer(csv_path, new_csv_path):

    df = pd.read_csv(csv_path).dropna()

    # from wgs 1984 to WGS 1984 Web Mercator (auxiliary sphere)
    wgs84 = Transformer.from_proj(4326, 3857, always_xy=True)

    # Calculate new points
    new_points = []
    for index, row in df.iterrows():
        x, y = wgs84.transform(row['Longitude_assist'], row['Latitude_assist'])
        x1, y1 = wgs84.transform(row['Longitude'], row['Latitude'])

        # Calculate new point
        new_x, new_y = coord_new_points(x, y, x1, y1, row['DistanceAverage'])
        new_points.append({'IMG':row['IMG'], 'CLS':row['CLS'], 'LON_ORIGIN':row['Longitude'], 'LAT_ORIGIN':row['Latitude'], 'LON': new_x, 'LAT':new_y})

    new_df = pd.DataFrame(new_points)
    new_df.to_csv(new_csv_path, index=False)
print ('hello2')

hello2


In [ ]:
# CONFIGURATION
switch_train = True   # Set to True to train, False to skip
switch_pred = False   # ✅ Set to True AFTER training completes to run prediction

if switch_train:
    # Load pretrained YOLOv8 model
    model = YOLO(YOLO_pretrained)

    # Train
    results = model.train(
        data=f'{DATA_DIR}/data.yaml',
        epochs=epochs,
        save_period=save_period,
        device=device,
        optimizer=optimizer,
        lr0=lr0,
        cos_lr=cos_lr,
        patience=patience,   # ✅ Early stopping added
        project=project_dir, # ✅ Fixed: was 'project' (conflicts with Roboflow project var)
        name=name
    )
    print('✅ Training complete! Best weights saved to:', MODEL_DIR)

if switch_pred:
    # ⚠️ Make sure switch_train = False and training is done before running this
    os.makedirs(os.path.join(ROOT, 'predi'), exist_ok=True)

    # PREDICTION
    prediction(
        model_dir=MODEL_DIR,
        img_dir=f'{TEST_DIR}/images',
        pred_dir=os.path.join(ROOT, 'predi'),
        switch_saveImg=True
    )

    # DISTANCE
    distance(
        pred_dir=os.path.join(ROOT, 'predi'),
        intercroppedCLS=4,
        intercropped_thrshld=0
    )

    # BEARING
    bearing(
        os.path.join(ROOT, 'predi', 'PRED_DIST_CLS_unique.csv'),
        os.path.join(ROOT, 'Maizecoord2026.csv'),
        os.path.join(ROOT, 'predi')
    )

    # FINAL COORDINATES
    coord_infer(
        os.path.join(ROOT, 'predi', 'bearing.csv'),
        os.path.join(ROOT, 'predi', 'label.csv')
    )

Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/Crop-Detection-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=detect, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print("Total VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)


NVIDIA A100-SXM4-80GB
Total VRAM (GB): 85.094825984


In [ ]:
import shutil

shutil.make_archive(
    "/content/crop_detection_results",
    "zip",
    "/content/Crop-Detection-1"
)

'/content/crop_detection_results.zip'

In [ ]:
from google.colab import files
files.download("/content/crop_detection_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>